In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [8]:
import cv2
import os
import polars as pl
import pickle
import numpy as np
from tqdm import tqdm
from google.colab import drive
import requests
import sys

Procesamiento de imagenes para deteccion de rostros con tecnologia DNN (Deep Neural Network) con un modelo Basado en SSD + ResNet.
A partir de imágenes a color, se convierte a gris después del recorte, Redimensionado	Resize a (64x64) y se ajustan coordenadas para ampliar el rostro detectado.

El Manejo de datos de los datos es mediante sea una estructura basada en subcarpetas por etiquetas (personas), como metodo de manejo de errores se ignoran imágenes no válidas, y se las guarda como no detectadas en un carpeta aparte. Para guardar los datos, se crea un DataFrame con Polars y se guarda como Pickle.

# --- Rutas y URLs de configuración ---

In [9]:
# Directorio donde se guardarán los modelos en Google Drive
MODEL_DIR = "/content/drive/MyDrive/OpenCV_DNN"

# Rutas completas a los archivos de modelo
PROTOTXT_PATH = os.path.join(MODEL_DIR, "deploy.prototxt")
CAFFE_MODEL_PATH = os.path.join(MODEL_DIR, "res10_300x300_ssd_iter_140000.caffemodel")

# URLs de descarga de los modelos
PROTOTXT_URL = "https://raw.githubusercontent.com/opencv/opencv/master/samples/dnn/face_detector/deploy.prototxt"
CAFFE_MODEL_URL = "https://github.com/opencv/opencv_3rdparty/raw/dnn_samples_face_detector_20170830/res10_300x300_ssd_iter_140000.caffemodel"

# Rutas para los datos
INPUT_DATA_PATH = "/content/drive/MyDrive/DMA/Caras"
OUTPUT_PKL_PATH = "/content/drive/MyDrive/DMA/datos_procesados/rostros_dataset.pkl"
NO_DETECTION_PATH = "/content/drive/MyDrive/DMA/datos_procesados/no_detectadas"

# Parámetros de procesamiento
PADDING_FACTOR = 0.2        # Factor de Relleno: Controla cuánto "espacio adicional" se incluye alrededor del rostro detectado al recortarlo en la función process_images_for_face_detection.
TARGET_SIZE = (64, 64)      # Tamaño de las imagenes redimensionadas

# --- Funciones ---

In [10]:
def download_file(url, destination_path):
    """Descarga un archivo de una URL a una ruta local."""
    print(f"Descargando {os.path.basename(destination_path)}...")
    try:
        response = requests.get(url, stream=True)
        response.raise_for_status()  # Lanza un error si la solicitud falla
        with open(destination_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        print("Descarga completa.")
        return True
    except requests.exceptions.RequestException as e:
        print(f"Error al descargar {url}: {e}")
        return False

def setup_models():
    """
    Verifica si los modelos existen en Drive y los descarga si no es así.
    """
    os.makedirs(MODEL_DIR, exist_ok=True)

    # Descargar el archivo .prototxt si no existe
    if not os.path.exists(PROTOTXT_PATH):
        if not download_file(PROTOTXT_URL, PROTOTXT_PATH):
            sys.exit("No se pudo descargar el archivo de configuración. Saliendo del script.")

    # Descargar el archivo .caffemodel si no existe
    if not os.path.exists(CAFFE_MODEL_PATH):
        if not download_file(CAFFE_MODEL_URL, CAFFE_MODEL_PATH):
            sys.exit("No se pudo descargar el archivo del modelo. Saliendo del script.")

def load_dnn_model(prototxt_path, caffe_model_path):
    """
    Carga el modelo DNN de Caffe para la detección de rostros.
    """
    try:
        net = cv2.dnn.readNetFromCaffe(prototxt_path, caffe_model_path)
        return net
    except cv2.error as e:
        print(f"Error al cargar el modelo DNN: {e}")
        return None

def process_images_for_face_detection(
    input_path, output_pkl_path, prototxt_path, caffe_model_path,
    no_detection_path, padding_factor=0.2, target_size=(64, 64)
):
    """
    Procesa un conjunto de imágenes para detectar, recortar, redimensionar y guardar rostros.
    """
    os.makedirs(no_detection_path, exist_ok=True)
    os.makedirs(os.path.dirname(output_pkl_path), exist_ok=True)

    net = load_dnn_model(prototxt_path, caffe_model_path)
    if net is None:
        return

    data = []

    labels = [label for label in os.listdir(input_path) if os.path.isdir(os.path.join(input_path, label))]

    for label in labels:
        label_path = os.path.join(input_path, label)
        image_files = [f for f in os.listdir(label_path) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp'))]

        for filename in tqdm(image_files, desc=f"Procesando {label}"):
            file_path = os.path.join(label_path, filename)

            try:
                image = cv2.imread(file_path)
                if image is None:
                    continue

                (h, w) = image.shape[:2]
                blob = cv2.dnn.blobFromImage(cv2.resize(image, (300, 300)), 1.0, (300, 300), (104.0, 177.0, 123.0))

                net.setInput(blob)
                detections = net.forward()

                face_detected = False
                for i in range(0, detections.shape[2]):
                    confidence = detections[0, 0, i, 2]

                    if confidence > 0.5:
                        box = detections[0, 0, i, 3:7] * np.array([w, h, w, h])
                        (startX, startY, endX, endY) = box.astype("int")

                        face_w = endX - startX
                        face_h = endY - startY
                        pad_x = int(face_w * padding_factor)
                        pad_y = int(face_h * padding_factor)

                        padded_startX = max(0, startX - pad_x)
                        padded_startY = max(0, startY - pad_y)
                        padded_endX = min(w, endX + pad_x)
                        padded_endY = min(h, endY + pad_y)

                        face_roi = image[padded_startY:padded_endY, padded_startX:padded_endX]

                        gray_face = cv2.cvtColor(face_roi, cv2.COLOR_BGR2GRAY)
                        resized_face = cv2.resize(gray_face, target_size, interpolation=cv2.INTER_AREA)

                        flat_face = resized_face.flatten()

                        data.append({
                            "imagen_data": flat_face,
                            "etiqueta": label,
                            "nombre_original": filename
                        })

                        face_detected = True
                        break

                if not face_detected:
                    cv2.imwrite(os.path.join(no_detection_path, filename), image)

            except Exception as e:
                pass

    if data:
        # Convert Polars DataFrame to Pandas DataFrame for pickling
        df_pl = pl.DataFrame(data)
        df_pd = df_pl.to_pandas()
        with open(output_pkl_path, 'wb') as f:
            pickle.dump(df_pd, f)
        print(f"DataFrame guardado con éxito en: {output_pkl_path}")
    else:
        print("No se encontraron rostros para procesar.")

# 1.Configura y descarga los modelos si es necesario

In [11]:
setup_models()

# 2. Ejecuta el procesamiento de imágenes

In [12]:
process_images_for_face_detection(
    input_path=INPUT_DATA_PATH,
    output_pkl_path=OUTPUT_PKL_PATH,
    prototxt_path=PROTOTXT_PATH,
    caffe_model_path=CAFFE_MODEL_PATH,
    no_detection_path=NO_DETECTION_PATH,
    padding_factor=PADDING_FACTOR,
    target_size=TARGET_SIZE
)

Procesando José: 100%|██████████| 26/26 [00:19<00:00,  1.30it/s]


DataFrame guardado con éxito en: /content/drive/MyDrive/DMA/datos_procesados/rostros_dataset.pkl
